# CS224N · Lecture 16 — ConvNets for NLP and Tree Recursive Neural Networks

**Slides:** [cs224n-spr2024-lecture16-CNN-TreeRNN.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture16-CNN-TreeRNN.pdf) (Christopher Manning)  
**Prerequisites:** L3 (backprop), L5–6 (RNNs), L8 (Transformers).

Two architectures that shaped NLP before Transformers took over, each built on an idea that's still everywhere:
* **Convolutions:** compute a vector for *every* window of $k$ words, then pool. (Convolutions are back in modern sequence models: the state space models in §6 are giant convolutions.)
* **Tree-recursive networks:** build phrase meanings by **composition** along a syntactic tree.

| § | Topic | You will implement / verify |
|---|---|---|
| 1 | From RNNs to CNNs: what a 1D convolution over text computes | The slides' worked example, reproduced number for number |
| 2 | Padding, channels, pooling (max, average, k-max, local), stride, dilation | All of them, in NumPy |
| 3 | **Kim (2014)**: a single-layer CNN for sentence classification | CNN from scratch (gradient-checked) vs. a bag-of-vectors baseline, on a task with negation |
| 4 | The word-vector fine-tuning pitfall; multichannel input | Measured |
| 5 | Model comparison; **BatchNorm vs. LayerNorm**; size-1 convolutions; very deep CNNs | BN's batch dependence; 1×1 conv = per-position linear layer |
| 6 | 🔎 State space models: a recurrence that *is* a convolution | Recurrent and convolutional views, verified equal |
| 7 | **Tree recursive neural networks**, greedy parsing, **RNTNs**, sentiment | Composition and greedy parsing; the RNTN's tensor layer |
| 8 | PyTorch: Kim's CNN | — |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, time, collections

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)

def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

---
## 1. From RNNs to convolutional neural nets

* **Recurrent neural nets cannot capture phrases without prefix context**: the RNN's vector for *"the ceremony"* in *"Monáe walked into the ceremony"* is really a vector for the whole prefix.
* They often **capture too much of the last words** in the final vector (e.g., a softmax for word prediction is usually computed from the last step).

**The main CNN idea:** what if we compute vectors for **every possible word subsequence of a certain length**? For *"tentative deal reached to keep government open"*, compute vectors for: *tentative deal reached*, *deal reached to*, *reached to keep*, *to keep government*, *keep government open*. **Regardless of whether the subsequence is grammatical** or a natural linguistic constituent (not very linguistically or cognitively plausible), then group them afterwards (pooling).

### 1.1 What is a convolution anyway?
1D discrete convolution, generally: $(f * g)[n] = \sum_{m=-M}^{M}f[n-m]\,g[m]$. Convolution is classically used to extract features from images (position-invariant identification; see CS231N). (Deep-learning "convolution" is technically **cross-correlation**, since the filter isn't flipped; it makes no difference when the filter is learned.)

### 1.2 A 1D convolution for text: the slides' example
Each word has a 4-dimensional embedding (one row). A filter of size 3 is a $3\times4$ matrix; slide it over every window of 3 words, take the elementwise product with the window, and sum to one number. Then add a bias and apply a non-linearity.

In [ ]:
words = ["tentative", "deal", "reached", "to", "keep", "government", "open"]
X = np.array([[0.2, 0.1, -0.3, 0.4], [0.5, 0.2, -0.3, -0.1], [-0.1, -0.3, -0.2, 0.4], [0.3, -0.3, 0.1, 0.1],
              [0.2, -0.3, 0.4, 0.2], [0.1, 0.2, -0.1, -0.1], [-0.4, -0.4, 0.2, 0.3]])
F1 = np.array([[3, 1, 2, -3], [-1, 2, 1, -3], [1, 1, -1, 1]], float)       # the slides' filter of size 3

def conv1d(X, filters, padding=0, stride=1, dilation=1):
    # X: (T, d); filters: (n_filters, k, d). Returns (positions, n_filters) and window labels.
    if padding:
        X = np.vstack([np.zeros((padding, X.shape[1])), X, np.zeros((padding, X.shape[1]))])
    k = filters.shape[1]; span = dilation * (k - 1) + 1
    starts = range(0, len(X) - span + 1, stride)
    out = np.array([[np.sum(f * X[s:s + span:dilation]) for f in filters] for s in starts])
    return out, list(starts)

out, starts = conv1d(X, F1[None])
abbrev = [w[0] for w in words]
for s, v in zip(starts, out[:, 0]):
    print(f"{','.join(abbrev[s:s + 3]):6s} {v:5.1f}")
print("slide values: -1.0, -0.5, -3.6, -0.2, 0.3")

---
## 2. Padding, channels, pooling, stride, dilation

* **Padding:** add zero vectors ∅ at the ends so every word gets a window centered on it (padding = 1 keeps the output length equal to the input length). Padding = 2 gives a **"wide convolution"**.
* **Multiple filters ("channels"):** apply 3 filters → 3 numbers per window; each filter can learn to detect a different kind of phrase.
* **Pooling over time:** summarize a variable-length sequence of window features into a fixed-size vector. **Max pooling** ("is this feature present anywhere?") or **average pooling**.
* Other notions: **stride** (skip windows), **local max pooling** (pool within small neighborhoods, halving the length), **k-max pooling** (keep the top $k$ values per filter, in order), **dilation** (spread the filter's taps apart to see a wider context with the same number of parameters).

All with the slides' three filters:

In [ ]:
F2 = np.array([[1, 0, 0, 1], [1, 0, -1, -1], [0, 1, 0, 1]], float)
F3 = np.array([[1, -1, 2, -1], [1, 0, -1, 3], [0, 2, 2, 1]], float)
filters = np.stack([F1, F2, F3])
padded_words = ["∅"] + abbrev + ["∅"]

H, starts = conv1d(X, filters, padding=1)
print("padding = 1, three filters:")
for s, row in zip(starts, H):
    print(f"  {','.join(padded_words[s:s + 3]):6s} {row}")
print(f"max pooling over time    : {H.max(0)}     (slide: 0.3 1.6 1.4)")
print(f"average pooling over time: {H.mean(0)}   (slide: -0.87 0.26 0.53)")

Hs, st = conv1d(X, filters, padding=1, stride=2)
print("\nstride = 2:", [",".join(padded_words[s:s + 3]) for s in st]); print(Hs)
Hp = np.vstack([H, np.full((1, 3), -np.inf)])                         # pad with -inf to an even length
print("local max pooling (size 2, stride 2):\n", Hp.reshape(-1, 2, 3).max(1))
k = 2
idx = np.sort(np.argsort(-H, axis=0)[:k], axis=0)                     # top-k positions, kept in original order
print(f"{k}-max pooling over time:\n", np.take_along_axis(H, idx, 0))
Hd, st = conv1d(X, filters, padding=1, dilation=2)
print("dilation = 2: windows", [",".join(padded_words[s:s + 5:2]) for s in st])

In PyTorch (the slides' snippet), note the **channels-first** layout: `Conv1d` expects `(batch, embedding_dim, seq_len)`:
```python
input = torch.randn(batch_size, word_embed_size, seq_len)
conv1 = Conv1d(in_channels=word_embed_size, out_channels=3, kernel_size=3)   # can add: padding=1
hidden1 = conv1(input)
hidden2 = torch.max(hidden1, dim=2)   # max pool over time
```

---
## 3. A single-layer CNN for sentence classification (Kim 2014)

**Goal:** sentence classification, mainly positive or negative sentiment, and other tasks like subjective vs. objective, or question type (about a person, location, number, …). A simple use of **one convolutional layer and max pooling**:
* Word vectors $\mathbf{x}_i\in\mathbb{R}^k$; a sentence $\mathbf{x}_{1:n} = \mathbf{x}_1\oplus\mathbf{x}_2\oplus\dots\oplus\mathbf{x}_n$ (concatenated).
* A convolutional filter $\mathbf{w}\in\mathbb{R}^{hk}$ over a window of $h$ words (h = 2, 3, or 4): $c_i = f(\mathbf{w}^\top\mathbf{x}_{i:i+h-1} + b)$, giving a **feature map** $\mathbf{c} = [c_1, \dots, c_{n-h+1}]$.
* **Max-over-time pooling:** $\hat c = \max\{\mathbf{c}\}$: capture the most important activation. Because of max pooling, the length of $\mathbf{c}$ can vary.
* Use **many filters** (Kim: 100 feature maps each of sizes 3, 4, 5), concatenate the pooled values into $\mathbf{z} = [\hat c_1, \dots, \hat c_m]$, apply **dropout** to $\mathbf{z}$, then a simple **softmax** layer.

### 3.1 A task where word order matters: negation
Synthetic movie-review sentences with **one sentiment adjective**; the label flips if the adjective is **immediately** preceded by *not* (*"not good"* is negative, *"not bad"* positive). Half the sentences also contain a **distractor** *not* elsewhere (*"it was not my idea , but the plot was good"*). A bag-of-words model sees *not* + *good* in both cases and can't tell them apart; a width-2 convolution can see the bigram *not good*.

In [ ]:
POS, NEG = ["good", "great", "fun", "moving"], ["bad", "dull", "boring", "slow"]
FILL = ["the", "movie", "plot", "was", "really", "acting", "it", "is", "and", "my", "idea", "but", "quite"]
SENT_LEN = 10

def review(rng):
    words_ = [rng.choice(FILL) for _ in range(SENT_LEN)]
    positive = rng.random() < 0.5
    adj = rng.choice(POS if positive else NEG)
    negated = rng.random() < 0.5
    pos = int(rng.integers(3, SENT_LEN))
    words_[pos] = adj
    if negated:
        words_[pos - 1] = "not"
    if rng.random() < 0.5:                                    # a distractor "not" that is NOT next to the adjective
        d = int(rng.integers(0, pos - 2)); words_[d] = "not"
    return words_, int(positive != negated)                    # label 1 = positive

r = np.random.default_rng(0)
data = [review(r) for _ in range(4000)]
vocab = sorted({w for s, _ in data for w in s}); w2i = {w: i for i, w in enumerate(vocab)}
Xids = np.array([[w2i[w] for w in s] for s, _ in data]); Y = np.array([y for _, y in data])
tr, te = slice(0, 3000), slice(3000, 4000)
for s, y in data[:4]:
    print(("+ " if y else "- ") + " ".join(s))

In [ ]:
D_EMB, N_FILT, WIDTHS = 16, 8, (2, 3)

def init_cnn(seed=0):
    r_ = np.random.default_rng(seed)
    p = {"E": r_.normal(0, 0.3, (len(vocab), D_EMB)), "U": r_.normal(0, 0.1, (N_FILT * len(WIDTHS), 2)), "c": np.zeros(2)}
    for h in WIDTHS:
        p[f"W{h}"] = r_.normal(0, 1 / np.sqrt(h * D_EMB), (h * D_EMB, N_FILT)); p[f"b{h}"] = np.zeros(N_FILT)
    return p

def windows(E_sent, h):                                         # (B, T, d) -> (B, T-h+1, h*d): unfold
    T = E_sent.shape[1]
    return np.concatenate([E_sent[:, i:T - h + 1 + i] for i in range(h)], axis=2)

def cnn_forward_backward(p, ids, y, drop=0.0, rng_=None, need_grad=True):
    B = len(ids); Es = p["E"][ids]
    feats, caches = [], []
    for h in WIDTHS:
        Wn = windows(Es, h); Z = Wn @ p[f"W{h}"] + p[f"b{h}"]; A = np.maximum(Z, 0)    # (B, T-h+1, F)
        am = A.argmax(1)                                                                  # max over time
        feats.append(np.take_along_axis(A, am[:, None, :], 1)[:, 0]); caches.append((Wn, Z, am))
    z = np.concatenate(feats, 1)
    mask = (rng_.random(z.shape) >= drop) / (1 - drop) if (drop and need_grad) else np.ones_like(z)
    P = softmax((z * mask) @ p["U"] + p["c"])
    loss = -np.log(P[np.arange(B), y] + 1e-12).mean()
    if not need_grad:
        return loss, P
    G = P.copy(); G[np.arange(B), y] -= 1; G /= B
    g = {"U": (z * mask).T @ G, "c": G.sum(0)}
    dz = (G @ p["U"].T) * mask
    dE_sent = np.zeros_like(Es)
    for j, h in enumerate(WIDTHS):
        Wn, Z, am = caches[j]
        dA = np.zeros_like(Z); np.put_along_axis(dA, am[:, None, :], dz[:, j * N_FILT:(j + 1) * N_FILT][:, None, :], 1)  # max routes
        dZ = dA * (Z > 0)
        g[f"W{h}"] = Wn.reshape(-1, Wn.shape[-1]).T @ dZ.reshape(-1, N_FILT); g[f"b{h}"] = dZ.sum((0, 1))
        dWn = dZ @ p[f"W{h}"].T                                                           # (B, T-h+1, h*d)
        for i in range(h):                                                                # fold back
            dE_sent[:, i:i + dWn.shape[1]] += dWn[:, :, i * D_EMB:(i + 1) * D_EMB]
    g["E"] = np.zeros_like(p["E"]); np.add.at(g["E"], ids, dE_sent)
    return loss, g

# gradient check
p_chk = init_cnn(1); ids_c, y_c = Xids[:4], Y[:4]
_, g_chk = cnn_forward_backward(p_chk, ids_c, y_c); worst = 0.0; r = np.random.default_rng(2)
for k_ in g_chk:
    for _ in range(5):
        i_ = (ids_c[0, int(r.integers(SENT_LEN))], int(r.integers(D_EMB))) if k_ == "E" else tuple(int(r.integers(s_)) for s_ in p_chk[k_].shape)
        old = p_chk[k_][i_]
        p_chk[k_][i_] = old + 1e-6; lp = cnn_forward_backward(p_chk, ids_c, y_c, need_grad=False)[0]
        p_chk[k_][i_] = old - 1e-6; lm_ = cnn_forward_backward(p_chk, ids_c, y_c, need_grad=False)[0]
        p_chk[k_][i_] = old
        worst = max(worst, abs((lp - lm_) / 2e-6 - g_chk[k_][i_]))
print(f"CNN gradient check: max |analytic - numerical| = {worst:.1e}")

**The baseline: a bag of vectors** (a "deep averaging network", Iyyer et al. 2015): average the word vectors, apply a ReLU layer, then softmax. The slides call it *"surprisingly good … for simple classification problems, especially if followed by a few ReLU layers."*

In [ ]:
def init_dan(seed=0):
    r_ = np.random.default_rng(seed)
    return {"E": r_.normal(0, 0.3, (len(vocab), D_EMB)), "W": r_.normal(0, 1 / np.sqrt(D_EMB), (D_EMB, 32)), "b": np.zeros(32),
            "U": r_.normal(0, 0.1, (32, 2)), "c": np.zeros(2)}

def dan_forward_backward(p, ids, y, need_grad=True, **_):
    B = len(ids); avg = p["E"][ids].mean(1); Z = avg @ p["W"] + p["b"]; A = np.maximum(Z, 0)
    P = softmax(A @ p["U"] + p["c"]); loss = -np.log(P[np.arange(B), y] + 1e-12).mean()
    if not need_grad:
        return loss, P
    G = P.copy(); G[np.arange(B), y] -= 1; G /= B
    g = {"U": A.T @ G, "c": G.sum(0)}; dZ = (G @ p["U"].T) * (Z > 0)
    g["W"], g["b"] = avg.T @ dZ, dZ.sum(0); davg = dZ @ p["W"].T
    g["E"] = np.zeros_like(p["E"]); np.add.at(g["E"], ids, np.repeat(davg[:, None, :] / ids.shape[1], ids.shape[1], 1))
    return loss, g

def train(fb, p, steps=1500, lr=3e-3, drop=0.0, trainable=None, seed=0):
    r_ = np.random.default_rng(seed); keys = trainable or list(p)
    m = {k_: np.zeros_like(p[k_]) for k_ in keys}; v = {k_: np.zeros_like(p[k_]) for k_ in keys}
    for t in range(1, steps + 1):
        idx = r_.integers(0, 3000, 64)
        _, g = fb(p, Xids[tr][idx], Y[tr][idx], drop=drop, rng_=r_)
        for k_ in keys:
            m[k_] = 0.9 * m[k_] + 0.1 * g[k_]; v[k_] = 0.999 * v[k_] + 0.001 * g[k_] ** 2
            p[k_] -= lr * (m[k_] / (1 - 0.9 ** t)) / (np.sqrt(v[k_] / (1 - 0.999 ** t)) + 1e-8)
    return p

def accuracy(fb, p, subset=te, mask=None):
    _, P = fb(p, Xids[subset], Y[subset], need_grad=False); pred = P.argmax(1)
    return np.mean(pred == Y[subset]) if mask is None else np.mean((pred == Y[subset])[mask])

has_distractor = np.array([("not" in s[:max(i for i, w in enumerate(s) if w in POS + NEG) - 1])
                           for s, _ in data[3000:]])
t0 = time.time()
p_dan = train(dan_forward_backward, init_dan())
p_cnn = train(cnn_forward_backward, init_cnn(), drop=0.3)
for name, fb, p in [("bag of vectors (DAN)", dan_forward_backward, p_dan), ("CNN (widths 2,3, max pool)", cnn_forward_backward, p_cnn)]:
    print(f"{name:28s} test accuracy {accuracy(fb, p):.1%}   without distractor {accuracy(fb, p, mask=~has_distractor):.1%}"
          f"   with distractor 'not' {accuracy(fb, p, mask=has_distractor):.1%}")
print(f"({time.time() - t0:.0f}s)")

In [ ]:
# Which bigrams does the strongest width-2 filter respond to?
W2 = p_cnn["W2"]
best_filter = np.argmax(np.abs(p_cnn["U"][:N_FILT, 1] - p_cnn["U"][:N_FILT, 0]))
scores = {}
for a in ["not", "the", "really"]:
    for b_ in POS + NEG:
        x = np.concatenate([p_cnn["E"][w2i[a]], p_cnn["E"][w2i[b_]]])
        scores[f"{a} {b_}"] = max(0, x @ W2[:, best_filter] + p_cnn["b2"][best_filter])
top = sorted(scores.items(), key=lambda kv: -kv[1])[:6]
print(f"bigrams that most activate width-2 filter #{best_filter}:", [(k_, round(v_, 2)) for k_, v_ in top])
print("the same filter on negated vs. un-negated adjectives:",
      {k_: round(scores[k_], 2) for k_ in ["not great", "the great", "not good", "the good", "not bad", "the bad"]})

The bag of vectors is stuck around 75%. Whenever a sentence contains *not*, it can't tell whether that *not* negates the adjective (half of the *not*s are distractors elsewhere in the sentence), so in those cases it can only guess. That's why its accuracy is similar with and without a distractor. The CNN's bigram filters see the adjacent pair directly and get 100%. Its filters are interpretable phrase detectors. The one printed above fires for a positive adjective preceded by anything *except* "not", and stays silent for "not great": a learned "un-negated positive" detector.

### 3.2 Regularization, and being careful in comparisons
Kim used dropout on $\mathbf{z}$ (the slides' 2014 setup), plus a max-norm constraint on the weights. **Be careful of fine points in comparisons!** Kim reports dropout gives a **2–4% accuracy improvement**, but several compared-to systems came earlier and didn't use dropout (from 2012/2014), and would probably gain equally from it. Still, these were remarkable results from a simple architecture. The differences from L2's window model: **many filters and pooling**.

| Model (Kim 2014) | MR | SST-1 | SST-2 | Subj | TREC | CR | MPQA |
|---|---|---|---|---|---|---|---|
| CNN-rand (random word vectors) | 76.1 | 45.0 | 82.7 | 89.6 | 91.2 | 79.8 | 83.4 |
| CNN-static (word2vec, frozen) | 81.0 | 45.5 | 86.8 | 93.0 | 92.8 | 84.7 | 89.6 |
| CNN-non-static (word2vec, fine-tuned) | 81.5 | 48.0 | 87.2 | 93.4 | 93.6 | 84.3 | 89.5 |
| CNN-multichannel | 81.1 | 47.4 | 88.1 | 93.2 | 92.2 | 85.0 | 89.4 |
| RNTN (Socher et al. 2013, §7) | – | 45.7 | 85.4 | – | – | – | – |

---
## 4. A pitfall when fine-tuning word vectors

**Setting:** we're training a movie-review sentiment model on top of pretrained word vectors. In the training data we have *tedious* and *dull*; in the test data we have *plodding*. The pretrained vectors have all three close together. **What happens when we update the word vectors?** Words in the training data move around; other words stay where they were. *Tedious* and *dull* move toward "negative"; *plodding* is left behind, possibly on the wrong side of the decision boundary. **This can be bad!**

**A solution: multichannel input** (Kim 2014). Initialize with pretrained vectors (word2vec or GloVe) and start with **two copies**. Backprop into only one set, and keep the other **"static"**. Both channels are added to $c_i$ before max pooling: fine-tuning can improve vectors for the task, while unseen words keep their pretrained neighbors. (Alternatively, leave all word vectors where they are and only update the parameters above them.)

Let's measure it, with "pretrained" vectors where each test-only synonym starts near a training word:

In [ ]:
def pitfall_trial(seed, update_vectors, dim=8, noise=0.3, steps=300, lr_w=0.5, lr_v=2.0):
    r_ = np.random.default_rng(seed)
    base = r_.normal(size=(8, dim)); labels_ = np.array([1] * 4 + [0] * 4)       # 8 meanings: 4 positive, 4 negative
    synonyms = base + noise * r_.normal(size=(8, dim))                           # test-only words, pretrained near a training word
    V = base.copy(); w = r_.normal(size=dim)                                      # training words start at their pretrained vectors
    for _ in range(steps):
        for i in range(8):                                                       # only TRAINING words appear in training data
            err = 1 / (1 + np.exp(-V[i] @ w)) - labels_[i]
            g_w, g_v = err * V[i], err * w
            w -= lr_w * g_w
            if update_vectors:
                V[i] -= lr_v * g_v                                               # fine-tuning moves only the training words
    return np.mean(((V @ w) > 0) == labels_), np.mean(((synonyms @ w) > 0) == labels_), np.linalg.norm(V - base, axis=1).mean()

for upd in [False, True]:
    res = np.array([pitfall_trial(seed, upd) for seed in range(20)])
    print(f"{'fine-tune word vectors' if upd else 'static word vectors   '}: training words {res[:, 0].mean():.0%}, "
          f"test-only synonyms {res[:, 1].mean():.1%}, avg movement of training words {res[:, 2].mean():.2f}   (20 seeds)")

Both versions fit the training words (nearly) perfectly, but fine-tuning moves the training words (by more than the typical gap to their synonyms), and the classifier is fit to the *moved* vectors, so the unmoved test-only synonyms are misclassified more often. (Averaged over 20 random seeds, because a single small run is noisy; that's also a general lesson.) With pretrained Transformers (L9) the same issue appears as **catastrophic forgetting** of rarely seen knowledge during finetuning, one motivation for small learning rates, LoRA (L12), and mixing in general data.

---
## 5. Our growing toolkit, and some CNN machinery

### 5.1 Model comparison
* **Bag of vectors:** surprisingly good baseline for simple classification problems, especially if followed by a few ReLU layers (Deep Averaging Networks).
* **Window model:** good for single-word classification for problems that don't need wide context, e.g., POS, NER.
* **CNNs:** good for classification; need zero padding for shorter phrases; somewhat implausible/hard to interpret; **easy to parallelize on GPUs**; efficient and versatile.
* **Recurrent neural networks:** cognitively plausible (reading left to right); not best for classification (if just using the last state); much slower than CNNs; good for sequence tagging and classification, and language models; better with attention.
* **Transformers:** great for language models, great for sentence calculations; in general, *"still the best thing since sliced bread for all NLP problems."*
* **Vision Transformers** are taking over in vision, but some papers argue CNNs and Transformers have complementary advantages, and you can usefully use both.

### 5.2 Batch normalization (Ioffe & Szegedy 2015)
Often used in CNNs: transform the convolution output of a batch by scaling the activations to zero mean and unit variance (like the familiar Z-transform of statistics). It's related to LayerNorm (standard in Transformers), but **crucially different**:
* **LayerNorm** computes statistics across all feature dimensions **for each instance independently**.
* **BatchNorm** normalizes across all items in a batch (and positions) **for each feature independently**.

BatchNorm makes models much less sensitive to parameter initialization (outputs are automatically rescaled) and tends to make learning rates easier to tune. In PyTorch: `nn.BatchNorm1d`. But its output for one example **depends on the other examples in the batch**, and it needs running statistics at test time:

In [ ]:
def batchnorm(Xb, eps=1e-5):                    # normalize each feature over the batch
    return (Xb - Xb.mean(0)) / np.sqrt(Xb.var(0) + eps)
def layernorm(Xb, eps=1e-5):                    # normalize each example over its features
    return (Xb - Xb.mean(1, keepdims=True)) / np.sqrt(Xb.var(1, keepdims=True) + eps)

r = np.random.default_rng(0)
x = r.normal(size=(1, 6))
batch_a = np.vstack([x, r.normal(size=(7, 6))]); batch_b = np.vstack([x, 5 + 3 * r.normal(size=(7, 6))])
print("same example, two different batches:")
print("  BatchNorm output :", batchnorm(batch_a)[0].round(2), "vs", batchnorm(batch_b)[0].round(2))
print("  LayerNorm output :", layernorm(batch_a)[0].round(2), "vs", layernorm(batch_b)[0].round(2))
print("BatchNorm with a batch of 1 (e.g., generating one sequence):", batchnorm(x).round(2), "<- all zeros: useless")

That batch dependence is why Transformers use LayerNorm: sequences have variable lengths, batches may be tiny, and autoregressive generation processes one token at a time.

### 5.3 Size-1 convolutions (Lin, Chen & Yan 2013, "Network in Network")
Does this concept make sense?! **Yes.** A size-1 convolution (`kernel_size=1`) is a **fully connected linear layer across channels, applied at every position**. It maps many channels to fewer (or more), adding extra layers with very few parameters (unlike a fully connected layer across the whole input, which adds tons). It's the same as the **per-position feed-forward layer in Transformers**:

In [ ]:
T_, C_in, C_out = 7, 4, 3
Xs = rng.normal(size=(T_, C_in)); W1x1 = rng.normal(size=(1, C_in, C_out))
conv_out = np.stack([Xs[t:t + 1].reshape(-1) @ W1x1.reshape(C_in, C_out) for t in range(T_)])   # kernel_size = 1
print("1x1 convolution == the same linear layer applied at every position:", np.allclose(conv_out, Xs @ W1x1[0]))
print(f"parameters: 1x1 conv {C_in * C_out}  vs  fully connected over the whole sequence {T_ * C_in * T_ * C_out}")

### 5.4 Very deep convolutional networks for text classification (Conneau, Schwenk, LeCun & Barrault, EACL 2017)
* **Starting point:** sequence models (LSTMs) had been very dominant in NLP; CNNs and attention were used too, but none were very deep, unlike vision models. **What happens when we build a vision-like system for NLP?**
* The model works up **from the character level** (the desire for "NLP from scratch" from the raw signal): input $s$ = 1024 characters, 16-dim embeddings.
* It looks like a vision system (VGGNet / ResNet), unlike the typical deep-learning NLP systems of the time (and a bit more like a Transformer?).
* **Convolutional block:** two conv layers (size 3), each followed by **BatchNorm** and ReLU; padding preserves the length, or halves it with local pooling. **Each local pooling stage halves the temporal resolution and doubles the number of features.** The result has constant size, since text is truncated or padded.
* Evaluated on large text-classification datasets, much bigger than Kim's; depth helped (up to 29 layers, with residual connections for the deepest).

---
## 6. 🔎 Beyond the slides: state space models, a recurrence that *is* a convolution

The slides quote a student survey: *"I would love to see more content about more recent models like state space models."* Here's the key idea, which connects this lecture's convolutions to L5–6's recurrences.

A (discrete, linear, time-invariant) **state space model** maps an input sequence $u_t$ to outputs $y_t$ through a hidden state:
$$h_t = \bar A h_{t-1} + \bar B u_t,\qquad y_t = C h_t$$
That's an **RNN with no nonlinearity**. Unroll it: $y_t = \sum_{k=0}^{t} C\bar A^k\bar B\,u_{t-k}$. So the output is a **convolution** of the input with the kernel $K = (C\bar B,\ C\bar A\bar B,\ C\bar A^2\bar B,\ \dots)$, a filter as long as the whole sequence.

Two views of the same model:
* **Convolutional view (training):** compute $K$ once, then convolve, in parallel over all positions (with FFTs, $O(T\log T)$). No sequential bottleneck, unlike an RNN.
* **Recurrent view (inference):** generate one step at a time with a fixed-size state: $O(1)$ memory per token, unlike a Transformer's growing KV cache (L8).

S4 (Gu et al. 2022) made this work for long sequences with a careful parameterization of $\bar A$ (so $\bar A^k$ neither explodes nor vanishes: L5–6's lesson). **Mamba** (Gu & Dao 2023) makes $\bar A, \bar B, C$ **input-dependent** ("selective"), which gives the model content-based gating like an LSTM's, but breaks the convolution view; it uses a parallel **scan** instead. Hybrids of attention and SSM/linear-recurrent layers are an active area for long-context models.

In [ ]:
r = np.random.default_rng(0)
N_state, T_len = 4, 50
A_bar = np.diag(r.uniform(0.6, 0.95, N_state))           # stable: eigenvalues inside the unit circle
B_bar, C_ = r.normal(size=(N_state, 1)), r.normal(size=(1, N_state))
u = r.normal(size=T_len)

h, y_rec = np.zeros((N_state, 1)), []
for t in range(T_len):                                   # recurrent view: one step at a time
    h = A_bar @ h + B_bar * u[t]; y_rec.append((C_ @ h).item())

K = np.array([(C_ @ np.linalg.matrix_power(A_bar, k) @ B_bar).item() for k in range(T_len)])   # the SSM kernel
y_conv = np.array([np.sum(K[:t + 1] * u[t::-1]) for t in range(T_len)])                         # convolutional view
y_fft = np.fft.irfft(np.fft.rfft(u, 2 * T_len) * np.fft.rfft(K, 2 * T_len))[:T_len]              # convolution via FFT
print("recurrence == convolution:", np.allclose(y_rec, y_conv), "  == FFT convolution:", np.allclose(y_rec, y_fft))
plt.figure(figsize=(6, 3)); plt.plot(K, "o-", ms=3); plt.xlabel("lag k"); plt.title("SSM kernel K_k = C A^k B (a learned, sequence-long filter)"); plt.show()

---
## 7. Tree recursive neural networks

### 7.1 Are languages recursive?
Cognitively somewhat debatable (you need to head to infinity). But **recursive structure is natural for describing language**: *[The person standing next to [the man from [the company that purchased [the firm that you used to work at]]]]*: a noun phrase containing a noun phrase containing a noun phrase... It's a very powerful **prior** for language structure (Penn Treebank trees, L4).

### 7.2 How should we map phrases into a vector space?
Use the **principle of compositionality**: the meaning (vector) of a phrase or sentence is determined by **(1) the meanings of its words** and **(2) the rules that combine them.** Then *"the country of my birth"* and *"the place where I was born"* should land near each other, as *Monday* lands near *Tuesday* (Socher, Manning & Ng, ICML 2011). Models in this section can **jointly learn parse trees and compositional vector representations**.

**Recursive vs. recurrent:** recursive neural nets provide representations for linguistic phrases, but they **require a tree structure**. Recurrent neural nets cannot capture phrases without prefix context, and often capture too much of the last words in a "phrase" vector.

### 7.3 A simple TreeRNN, and greedy parsing
The network takes **two candidate children's representations**, and outputs (1) the semantic representation if the two nodes are merged, and (2) a **score** of how plausible the new node would be:
$$p = \tanh\left(W\begin{bmatrix}c_1\\c_2\end{bmatrix} + b\right),\qquad \text{score} = U^\top p$$
with the **same $W$ at all nodes** of the tree. **Parsing a sentence greedily:** score every adjacent pair, merge the best-scoring pair into a new node, and repeat until one node remains. The score of a tree is the sum of the parsing-decision scores at each node: $s(x, y) = \sum_{n\in\text{nodes}(y)}s_n$. (Training uses a max-margin objective that pushes the correct tree's score above incorrect ones'.)

In [ ]:
d_t = 6
r = np.random.default_rng(0)
W_t, b_t, U_t = r.normal(0, 0.5, (d_t, 2 * d_t)), np.zeros(d_t), r.normal(size=d_t)
sent = "the cat sat on the mat".split()
leaf = {w: r.normal(size=d_t) for w in set(sent)}

def compose(c1, c2):
    p = np.tanh(W_t @ np.concatenate([c1, c2]) + b_t)
    return p, U_t @ p

def greedy_parse(words_):
    nodes = [(w, leaf[w]) for w in words_]; total = 0.0
    while len(nodes) > 1:
        cands = [compose(nodes[i][1], nodes[i + 1][1]) for i in range(len(nodes) - 1)]
        i = int(np.argmax([s_ for _, s_ in cands])); p, s_ = cands[i]; total += s_
        nodes = nodes[:i] + [(f"({nodes[i][0]} {nodes[i + 1][0]})", p)] + nodes[i + 2:]
        print(f"merge -> {nodes[i][0]:40s} score {s_:+.2f}")
    return nodes[0], total

(tree, vec), score = greedy_parse(sent)
print(f"\nfinal tree: {tree}   (total score {score:+.2f}; random weights, so this tree is arbitrary until trained)")

**Discussion: the simple TreeRNN.** It got decent results with a single layer (Socher, Manning & Ng, ICML 2011, best paper award)! But a single weight matrix could capture some things, not more complex, higher-order composition, or parsing long sentences. There is **no real interaction between the input words** (they're just concatenated and passed through one linear map), and the **composition function is the same for all syntactic categories**, punctuation, etc.

### 7.4 Recursive Neural Tensor Networks and sentiment (Socher, Perelygin, Wu, Chuang, Manning, Ng & Potts 2013)
**Beyond the bag of words: sentiment detection.** Sentiment is often thought "easy": detection accuracy for longer documents is ~90% (lots of *loved*, *great*, *impressed*, *marvelous*...). But sentences with negation and contrast are hard.

**The Stanford Sentiment Treebank:** **215,154 phrases labeled in 11,855 sentences**, a sentiment label for *every node* of every parse tree, so models can actually be trained and tested on compositions. A better dataset helped all models, but hard negation cases were still mostly incorrect: **we need a more powerful model!**

**The RNTN idea:** allow both **additive** and **mediated multiplicative** interactions of vectors, with a tensor $V^{[1:d]}\in\mathbb{R}^{2d\times2d\times d}$:
$$p = \tanh\left(\begin{bmatrix}c_1\\c_2\end{bmatrix}^\top V^{[1:d]}\begin{bmatrix}c_1\\c_2\end{bmatrix} + W\begin{bmatrix}c_1\\c_2\end{bmatrix}\right)$$
Each output dimension $k$ gets its own bilinear form $[c_1;c_2]^\top V^{[k]}[c_1;c_2]$, so one child can **modulate** how the other is interpreted (exactly what *not* does to *good*). The resulting vectors at every node feed a softmax classifier, and all weights train jointly by gradient descent.

In [ ]:
V_t = r.normal(0, 0.1, (d_t, 2 * d_t, 2 * d_t))            # one 2d x 2d bilinear form per output dimension

def rntn_compose(c1, c2):
    c = np.concatenate([c1, c2])
    bilinear = np.einsum("i,kij,j->k", c, V_t, c)           # [c1;c2]^T V^[k] [c1;c2] for each k
    return np.tanh(bilinear + W_t @ c + b_t)

# The multiplicative term lets one child change the effect of the other: test linearity in c2.
c1, c2a, c2b = r.normal(size=d_t), r.normal(size=d_t), r.normal(size=d_t)
def pre_act(f_bilinear, c1, c2):
    c = np.concatenate([c1, c2]); return (np.einsum("i,kij,j->k", c, V_t, c) if f_bilinear else 0) + W_t @ c
for name, bil in [("TreeRNN (additive only)", False), ("RNTN (+ tensor)", True)]:
    lhs = pre_act(bil, c1, c2a + c2b) - pre_act(bil, c1, c2a) - pre_act(bil, c1, c2b) + pre_act(bil, c1, np.zeros(d_t))
    print(f"{name:25s}: interaction term between children = {np.linalg.norm(lhs):.3f}  (0 = children don't interact)")
print("RNTN parent vector:", rntn_compose(c1, c2a).round(3))

**Results on the treebank (positive/negative):** sentence accuracy improves to **85.4%** (RNTN), vs. RNN, MV-RNN, and bigram Naive Bayes; and training on the full **treebank** labels (every phrase) helps all models compared with training on sentence labels only. The RNTN can capture constructions like **"X but Y"** (where Y dominates): **72%** accuracy, vs. MV-RNN 65%, biword NB 58%, RNN 54%. **Negation:** when negating negatives (*"not bad"*), positive activation should *increase*, and the RNTN is the model that gets this right.

**Not today, but see also:** **Tree-LSTMs** (Tai, Socher & Manning 2015), which work even better.

🔎 **Where these ideas went.** Tree-structured composition lost to Transformers, which need no parser and learn soft, implicit structure from data (L4's structural probes found syntax inside BERT). But the **Stanford Sentiment Treebank** (SST-2/SST-5) became a standard benchmark (it's in GLUE, L9), and **compositional generalization** (does a model combine known parts in new ways?) remains an open problem that the slides' COGS/ReCOGS pointers (L7) study.

---
## 8. Kim's CNN in PyTorch

In [ ]:
# [PyTorch] Kim (2014)-style CNN on the negation dataset.
import torch, torch.nn as nn, torch.nn.functional as F

class KimCNN(nn.Module):
    def __init__(self, vocab_size, emb=16, n_filters=8, widths=(2, 3), n_classes=2, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb)
        self.convs = nn.ModuleList(nn.Conv1d(emb, n_filters, kernel_size=w) for w in widths)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(n_filters * len(widths), n_classes)
    def forward(self, ids):                                   # ids: (B, T)
        x = self.emb(ids).transpose(1, 2)                     # (B, emb, T): channels first for Conv1d
        pooled = [F.relu(conv(x)).max(dim=2).values for conv in self.convs]   # max over time
        return self.out(self.drop(torch.cat(pooled, dim=1)))

torch.manual_seed(0)
model = KimCNN(len(vocab)); opt = torch.optim.Adam(model.parameters(), lr=3e-3)
Xt, Yt = torch.tensor(Xids), torch.tensor(Y)
for step in range(1500):
    idx = torch.randint(0, 3000, (64,))
    loss = F.cross_entropy(model(Xt[idx]), Yt[idx]); opt.zero_grad(); loss.backward(); opt.step()
model.eval()
with torch.no_grad():
    pred = model(Xt[3000:]).argmax(1).numpy()
print(f"PyTorch Kim CNN: test accuracy {np.mean(pred == Y[3000:]):.1%}, with distractor 'not' {np.mean((pred == Y[3000:])[has_distractor]):.1%}")

---
## 9. Summary

| Concept | Key point |
|---|---|
| 1D convolution over text | A vector (per filter) for every $k$-word window, grammatical or not |
| Padding, channels | Zero vectors at the ends; many filters = many phrase detectors |
| Pooling | Max ("is it anywhere?"), average, k-max, local; handles variable length |
| Stride, dilation | Skip windows; widen the receptive field cheaply |
| Kim (2014) | Embeddings → convolutions of widths 3/4/5 → max over time → dropout → softmax |
| Measured (toy) | Bag of vectors ~75% (can't tell which word *not* negates); CNN 100% with bigram phrase detectors |
| Fine-tuning pitfall | Only training words move; multichannel (static + tuned) input |
| BatchNorm vs. LayerNorm | BN: per feature across the batch (batch-dependent); LN: per example (Transformers) |
| 1×1 convolution | Per-position linear layer across channels (= Transformer FFN) |
| 🔎 SSMs | Linear recurrence = long convolution; parallel training, constant-memory inference |
| TreeRNN | $p = \tanh(W[c_1;c_2] + b)$, same $W$ everywhere; greedy parsing by scores |
| RNTN | Adds a bilinear tensor term so children interact multiplicatively; handles negation, *X but Y* |

### Exercises
1. Add a **width-1** filter group to the CNN. Does it help or hurt on the negation task?
2. Replace max-over-time with **average** pooling in the CNN. Why does accuracy drop on long sentences with one key phrase?
3. Implement **multichannel** input (a frozen and a tuned copy of the embeddings summed before pooling) and rerun §4 with `steps=3000`.
4. Implement **k-max pooling** in the CNN and its backward pass (the gradient routes to the $k$ argmax positions).
5. Train the TreeRNN and the RNTN on synthetic trees like *(not (very good))*, with a sentiment label at every node. Does the RNTN learn negation better?
6. Make the SSM in §6 **selective**: let $\bar B$ depend on $u_t$. Show that the convolution view no longer matches the recurrence.

### References
* Kim 2014, *Convolutional Neural Networks for Sentence Classification*; Zhang & Wallace 2015, *A Sensitivity Analysis of (and Practitioners' Guide to) CNNs for Sentence Classification*.
* Kalchbrenner, Grefenstette & Blunsom 2014 (DCNN, k-max pooling); Collobert et al. 2011.
* Conneau, Schwenk, LeCun & Barrault 2017, *Very Deep Convolutional Networks for Text Classification*.
* Iyyer et al. 2015 (Deep Averaging Networks); Ioffe & Szegedy 2015 (BatchNorm); Lin, Chen & Yan 2013 (Network in Network).
* Socher, Manning & Ng 2011; Socher et al. 2013 (RNTN, Stanford Sentiment Treebank); Tai, Socher & Manning 2015 (Tree-LSTMs).
* Gu, Goel & Ré 2022 (S4); Gu & Dao 2023 (Mamba).